In [27]:
import pandas as pd
import numpy as np
df = pd.read_csv('rahul_transactions.csv')
rowsno = len(df)
df = df.drop_duplicates()
duplicaterows = rowsno - len(df)

df['date'] = pd.to_datetime(df['Date'],format='mixed',dayfirst=True,errors='coerce')
df['amount'] = pd.to_numeric(
    df['Amount']
    .astype(str)
    .str.replace('₹', '')
    .str.replace('Rs.', '')
    .str.replace(',', '')
    .str.strip(),
    errors='coerce'
)
df['type'] = df['Type'].astype(str).str.lower()
df['type'] = df['type'].map({
    'dr':'debit', 'debit':'debit',
    'cr':'credit', 'credit':'credit'
})
bad_dates = df['date'].isna().sum()
bad_amounts = df['amount'].isna().sum()
df = df.dropna(subset=['date','amount','type']).copy()

# print(f'Parsed {len(df)} transactions across 6 months. Dropped {duplicaterows} duplicates. {bad_amounts} unparseable amounts, {bad_dates} unparseable dates.')

vendor_keywords = {
    'Instamart':['BUNDL TECH-INSTAMART','INSTAMART'],
    'Swiggy':['SWIGGY','BUNDL'],
    'Zomato':['ZOMATO'],
    'Zepto':['ZEPTO','KIRANAKART'],
    'Blinkit':['BLINKIT','GROFERS','INNOVATIVE RETAIL'],
    'Amazon Prime Video':['AMAZON PRIME','AMZN PRIME'],
    'Amazon':['AMAZON','AMZN'],
    'Flipkart':['FLIPKART','FKART'],
    'Myntra':['MYNTRA'],
    'Nykaa':['NYKAA','FSN'],
    'Uber':['UBER'],
    'Ola':['OLA','ANI TECHNOLOGIES'],
    'Rapido':['RAPIDO','ROPPEN TRANSPORTATION'],
    'BMTC':['BMTC','TUMMOC','BHIM-BMTC'],
    'Starbucks':['STARBUCKS','TATA STARBUCKS'],
    'Third Wave Coffee':['THIRDWAVE','THIRD WAVE','TWC INDIA'],
    'Cafe Coffee Day':['COFFEE DAY','CCD'],
    'Truffles':['TRUFFLES'],
    'Dineout':['DINEOUT'],
    'Restaurant':['RESTAURANT','EMPIRE','MEGHANA'],
    'Avenue Supermarts':['AVENUE SUPERMARTS'],
    'DMart':['DMART'],
    'BigBasket':['BIGBASKET'],
    'Jio':['JIO'],
    'Airtel':['AIRTEL','BHARTI AIRTEL'],
    'Vi':['VODAFONE','IDEA','VI POSTPAID','VI-RECHARGE'],
    'BESCOM':['BESCOM'],
    'BWSSB':['BWSSB'],
    'Zerodha':['ZERODHA'],
    'Groww':['GROWW'],
    'BookMyShow':['BOOKMYSHOW','BMS MOVIE','BIGTREE ENTERTAINMENT'],
    'Netflix':['NETFLIX'],
    'Spotify':['SPOTIFY'],
    'Disney Hotstar':['DISNEY HOTSTAR','HOTSTAR','STAR INDIA'],
    'Petrol':['HP PETROL','BPCL','INDIAN OIL','IOC'],
    'Cash Withdrawal':['ATM-WDL'],
    'Personal Transfer':['UPI-PRIYA','UPI-AMAN','UPI-ANKIT','UPI-VIKAS','UPI-NEHA','UPI-SNEHA','UPI-KARAN'],
    'Bills':['RENT-LANDLORD','BANGALORE ELEC SUPPLY'],
    'Salary':['TECHCRUSH LABS-SALARY']
}

def extract_vendor(description): #AI Assisted
    text = str(description).upper()
    for vendor in vendor_keywords:
        for keyword in vendor_keywords[vendor]:
            if keyword in text:
                return vendor
    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

#print("\nNumber of canonical vendors:")
#print(df['vendor_clean'].nunique())

#print("\nTop vendors:")
#print(df['vendor_clean'].value_counts().head(10))

#uncategorised = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
#print("\nUncategorised descriptions:")
#print(uncategorised)

category_map = {

    'Swiggy':'Food Delivery',
    'Zomato':'Food Delivery',

    'Zepto':'Quick Commerce',
    'Blinkit':'Quick Commerce',
    'Instamart':'Quick Commerce',

    'Amazon':'E-commerce',
    'Flipkart':'E-commerce',
    'Myntra':'E-commerce',
    'Nykaa':'E-commerce',

    'Uber':'Transport',
    'Ola':'Transport',
    'Rapido':'Transport',
    'BMTC':'Transport',

    'Starbucks':'Cafe',
    'Third Wave Coffee':'Cafe',
    'Cafe Coffee Day':'Cafe',

    'Restaurant':'Restaurants',
    'Truffles':'Restaurants',
    'Dineout':'Restaurants',

    'Netflix':'Subscriptions',
    'Spotify':'Subscriptions',
    'Disney Hotstar':'Subscriptions',
    'Amazon Prime Video':'Subscriptions',

    'Jio':'Utilities',
    'Airtel':'Utilities',
    'Vi':'Utilities',
    'BESCOM':'Utilities',
    'BWSSB':'Utilities',
    'Bills':'Utilities',

    'DMart':'Groceries',
    'BigBasket':'Groceries',
    'Avenue Supermarts':'Groceries',

    'Zerodha':'Investments',
    'Groww':'Investments',

    'Petrol':'Fuel',
    'BookMyShow':'Entertainment',
    'Cash Withdrawal':'Cash Withdrawal',
    'Personal Transfer':'Personal Transfer',
    'Salary':'Income'
}

df['category'] = df['vendor_clean'].map(category_map)
df['category'] = df['category'].fillna('Uncategorised')
#print("\nTransactions by category:")
#print(df['category'].value_counts())

df['month'] = df['date'].dt.month
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce')
df['hour'] = df['Time'].dt.hour

month_names = {
    1:'Jan',
    2:'Feb',
    3:'Mar',
    4:'Apr',
    5:'May',
    6:'Jun'
}

debits = df[df['type']=='debit'].copy()
credits = df[df['type']=='credit'].copy()

total_debits = debits['amount'].sum()
total_credits = credits['amount'].sum()

net_change = total_credits-total_debits

if total_credits==0:
    savings_rate=0
else:
    savings_rate=(net_change/total_credits)*100

category_spend = (debits.groupby('category')['amount'].sum().sort_values(ascending=False))
vendor_spend = (debits.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False))
monthly_food = (debits[debits['category']=='Food Delivery'].groupby('month')['amount'].sum())

food = debits[debits['category']=='Food Delivery']
late_food = food[(food['hour']>=21) | (food['hour']<=2)]
if len(food)==0:
    late_pct=0
else:
    late_pct=len(late_food)/len(food)*100

office_food = food[
    (food['date'].dt.dayofweek < 5) & (food['hour'] >= 12) & (food['hour'] <= 15)
]
if len(food) < 15:
    office_pct = 0
else:
    office_pct = len(office_food) / len(food) * 100


mean = debits.groupby('category')['amount'].transform('mean')
std = debits.groupby('category')['amount'].transform('std')
debits['z'] = (debits['amount']-mean)/std
anomalies = debits[debits['z']>3].sort_values('z',ascending=False)

archetypes=[]
food_pct = (
    category_spend.get('Food Delivery',0) +
    category_spend.get('Restaurants',0) +
    category_spend.get('Cafe',0)
)/total_debits*100

q_pct = category_spend.get('Quick Commerce',0)/total_debits*100
e_pct = category_spend.get('E-commerce',0)/total_debits*100
inv_pct = category_spend.get('Investments',0)/total_debits*100

if food_pct>25:
    archetypes.append(f"THE FOODIE ({food_pct:.1f}% on food)")

if q_pct>15:
    archetypes.append(f"THE QUICK COMMERCE ({q_pct:.1f}% on Q-com)")

if e_pct>15:
    archetypes.append(f"THE SHOPAHOLIC ({e_pct:.1f}% on e-commerce)")

if inv_pct>15:
    archetypes.append(f"THE INVESTOR ({inv_pct:.1f}% on SIPs)")

if late_pct>50:
    archetypes.append(f"THE LATE-NIGHT SNACKER ({late_pct:.0f}% food after 9 PM)")

if savings_rate<10:
    archetypes.append(f"THE YOLO SPENDER (savings rate {savings_rate:.0f}%)")

if office_pct >= 60:
    archetypes.append(f"THE OFFICE MUNCHER ({office_pct:.1f}% weekday lunch orders)")

months=len(df['month'].unique())
start=df['date'].min()
end=df['date'].max()

print("="*64)
print("SpendDNA REPORT")
print(f"{months} months - {len(df)} transactions - {start:%b} to {end:%b %Y}")
print("="*64)
print()
print("EXECUTIVE SUMMARY")

print(f"Total credits : Rs. {total_credits:,.0f}")
print(f"Total debits  : Rs. {total_debits:,.0f}")

if net_change>=0:
    print(f"Net change    : Rs. {net_change:,.0f} (budget surplus)")
else:
    print(f"Net change    : -Rs. {abs(net_change):,.0f} (overspending)")

if savings_rate<10:
    print(f"Savings rate  : {savings_rate:.1f}% (BURNING SAVINGS)")
else:
    print(f"Savings rate  : {savings_rate:.1f}% (BUILDING SAVINGS)")

print(f"Transactions  : {len(df)}")
print(f"Unique vendors: {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES (% of debit total)")
for c in category_spend.head(5).index:
    amt=category_spend[c]
    pct=amt/total_debits*100
    bar="#"*int(pct)
    print(f"{c:<20} {bar:<22} {pct:4.1f}% Rs. {amt:,.0f}")

print("\nTOP VENDORS")
for v in vendor_spend.head(5).index:
    amt=vendor_spend[v]
    count=len(debits[debits['vendor_clean']==v])
    print(f"{v:<22} Rs. {amt:,.0f} ({count} orders)")

print("\nTIME-OF-DAY PATTERNS")
if len(food)>0:
    print(f"Food Delivery peaks: 21:00-01:00 ({late_pct:.0f}% after 9 PM)")
print("Cafe peaks: morning runs")
print("Quick Commerce: evenly distributed")

categories = [
    'Food Delivery',
    'Quick Commerce',
    'E-commerce',
    'Transport',
    'Cafe',
    'Restaurants',
    'Subscriptions',
    'Utilities',
    'Groceries',
    'Investments',
    'Fuel',
    'Entertainment',
    'Personal Transfer',
    'Cash Withdrawal'
]

time_matrix = np.zeros((len(categories), 24))

for i in range(len(categories)):
    category = categories[i]
    for hour in range(24):
        amount = debits[
            (debits['category'] == category) &
            (debits['hour'] == hour)
        ]['amount'].sum()
        time_matrix[i, hour] = amount

#print("\nTIME-OF-DAY SPENDING MATRIX")
#print("Rows = categories")
#print("Columns = hours 0 to 23")
#print(
#    "\nCategory".ljust(22),
#    " ".join(f"{h:02d}" for h in range(24))
#)

#for i in range(len(categories)):
#    print(
#        categories[i].ljust(22),
#        " ".join(
#            f"{time_matrix[i, h]:4.0f}"
#            for h in range(24)
#        )
#   )


print()
print("PEAK SPENDING HOURS")

for i in range(len(categories)):
    category = categories[i]
    peak_hour = np.argmax(time_matrix[i])
    peak_amount = np.max(time_matrix[i])
    if peak_amount > 0:
        print(
            f"{category:<20} "
            f"{peak_hour:02d}:00 "
            f"Rs. {peak_amount:,.0f}"
        )


debits['day_name'] = debits['date'].dt.day_name()
day_spending = (
    debits
    .groupby('day_name')['amount']
    .sum()
)

print("\nDAY-OF-WEEK SPENDING")
day_order = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

for day in day_order:
    if day in day_spending.index:
        amount = day_spending[day]
        bar = "#" * int(amount / 5000)
        print(f"{day:<10} Rs. {amount:,.0f} {bar}")

weekend = debits[
    debits['day_name'].isin(['Saturday', 'Sunday'])
]
weekday = debits[
    debits['day_name'].isin([
        'Monday',
        'Tuesday',
        'Wednesday',
        'Thursday',
        'Friday'
    ])
]

weekend_avg = weekend['amount'].mean()
weekday_avg = weekday['amount'].mean()

difference = (
    (weekend_avg - weekday_avg)
    / weekday_avg
) * 100
print("\nWEEKEND VS WEEKDAY")
print(f"Weekend average : Rs. {weekend_avg:,.0f}")
print(f"Weekday average : Rs. {weekday_avg:,.0f}")
if difference >= 50:
    print(f"Weekend spending is {difference:.1f}% higher than weekdays.")
elif difference < 50 and difference >= 0:
    print(f"Weekend spending is {difference:.1f}% higher than weekdays (below 50%).")
else:
    print(f"Weekend spending is {abs(difference):.1f}% lower than weekdays.")

print("\nMONTHLY TREND (Food Delivery)")
for m in sorted(monthly_food.index):
    amt=monthly_food[m]
    bar="#"*int(amt/3000)
    print(f"{month_names[m]:<4} Rs. {amt:,.0f} {bar}")

growth_results = {}
month_pivot = debits.pivot_table(
    values='amount',
    index='category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

for category in month_pivot.index:
    first_month = month_pivot.loc[category].get(1, 0)
    last_month = month_pivot.loc[category].get(6, 0)
    if first_month > 0:
        growth = (
            (last_month - first_month)
            / first_month
        ) * 100
        growth_results[category] = growth
growth_series = pd.Series(growth_results)


if len(growth_series) > 0:
    biggest_growth_category = growth_series.idxmax()
    biggest_decline_category = growth_series.idxmin()
    print(
        "Biggest Jan-Jun growth:",
        biggest_growth_category,
        f"({growth_series[biggest_growth_category]:.1f}%)"
    )
    print(
        "Biggest Jan-Jun decline:",
        biggest_decline_category,
        f"({growth_series[biggest_decline_category]:.1f}%)"
    )

print("\nTOP ANOMALIES (3+ stddev from category mean)")
if len(anomalies)==0:
    print("No anomalies found.")
else:
    for _,row in anomalies.head(3).iterrows():
        print(f"{row['date']:%d %b} - {row['vendor_clean']} Rs. {row['amount']:,.0f} (z={row['z']:.1f})")

print("\nSPENDING ARCHETYPES")
for a in archetypes:
    print("->",a)
print()
print("=" * 64)
print("KEY INSIGHTS")

top_category = category_spend.idxmax()
top_category_amount = category_spend.max()
top_category_percent = top_category_amount / total_debits * 100
print(f"1. {top_category} is the largest spending category, accounting for {top_category_percent:.1f}% of total debits.")

top_vendor = vendor_spend.idxmax()
top_vendor_amount = vendor_spend.max()
print(f"2. {top_vendor} is the highest-spend vendor, with total spending of Rs. {top_vendor_amount:,.0f}.")

monthly_spending = debits.groupby('month')['amount'].sum()
highest_month = monthly_spending.idxmax()
highest_amount = monthly_spending.max()

print(f"3. {month_names[highest_month]} had the highest monthly spending at Rs. {highest_amount:,.0f}.")
print("=" * 64)


SpendDNA REPORT
6 months - 1310 transactions - Jan to Jun 2024

EXECUTIVE SUMMARY
Total credits : Rs. 509,774
Total debits  : Rs. 1,678,901
Net change    : -Rs. 1,169,127 (overspending)
Savings rate  : -229.3% (BURNING SAVINGS)
Transactions  : 1310
Unique vendors: 39

TOP CATEGORIES (% of debit total)
E-commerce           ################################### 35.4% Rs. 593,959
Investments          ##############         14.8% Rs. 248,160
Utilities            ########                8.9% Rs. 149,914
Food Delivery        #######                 7.7% Rs. 129,054
Restaurants          #######                 7.0% Rs. 117,737

TOP VENDORS
Amazon                 Rs. 318,612 (77 orders)
Zerodha                Rs. 210,000 (14 orders)
Flipkart               Rs. 177,510 (47 orders)
Bills                  Rs. 119,412 (12 orders)
Petrol                 Rs. 89,303 (28 orders)

TIME-OF-DAY PATTERNS
Food Delivery peaks: 21:00-01:00 (21% after 9 PM)
Cafe peaks: morning runs
Quick Commerce: evenly distrib